# Đề tài 5KH · LLM dự đoán đầu ra Python qua nhiều tệp

Notebook tạo các kho mã Python tổng hợp, chạy chương trình để lấy **đáp án chuẩn** và dùng vết thực thi để đánh dấu hàm liên quan cho từng đầu vào. Cả ba RQ sử dụng cùng các ca. Mô hình nhận mã nguồn và đầu vào, rồi trả về giá trị nguyên của lời gọi hàm.

| RQ | So sánh | Phép đo |
|---|---|---|
| RQ1 | Cùng mã và cùng số token, đặt nhóm hàm đã chạy ở gần đầu / giữa / cuối ngữ cảnh dài | Độ chính xác và chênh lệch ghép cặp theo từng ca |
| RQ2 | BM25 và BM25 ưu tiên hàm lân cận trên đồ thị lời gọi tĩnh, cùng giới hạn token | Tỷ lệ tìm đủ hàm, độ bao phủ hàm và độ chính xác |
| RQ3 | So sánh từng lỗi của BM25 và BM25 + đồ thị với kết quả khi cấp ngữ cảnh chuẩn theo vết thực thi | Số lỗi được sửa hoặc còn lại, chia theo độ bao phủ truy xuất ban đầu |

**Chạy trên Jupyter hoặc Colab** · Chọn **Run All** và nhập khóa API VTS qua ô nhập ẩn. Notebook đã cấu hình endpoint VTS với **Qwen/Qwen3-VL-8B-Instruct-FP8**, model nhỏ nhất theo tổng số tham số trong danh sách đã kiểm tra. Nếu chỉ cần tạo dữ liệu, kiểm tra thiết kế và xem các chỉ số truy xuất, đặt **RUN_LLM = False** ở ô cấu hình trước khi chạy. Lần thử đầu nên để 3 ca mỗi họ (54 yêu cầu), sau đó tăng lên 8 ca mỗi họ (144 yêu cầu) hoặc hơn. Mỗi ca có 6 cấu hình. Mỗi cấu hình gọi mô hình một lần.

Dữ liệu tổng hợp cho phép kiểm soát vị trí và tự động chấm điểm. Các ca cùng họ có chung cấu trúc, vì vậy kết quả mô tả các chương trình trong bộ dữ liệu này. Đồ thị tĩnh chỉ phân giải lời gọi đặt tên trực tiếp. Họ nested dùng một lời gọi qua getattr để khảo sát giới hạn này. Ngữ cảnh chuẩn giúp chẩn đoán lỗi có thể sửa bằng cách đổi ngữ cảnh. Nó cũng bỏ các đoạn gây nhiễu, nên tỷ lệ sửa lỗi không đo riêng tác động của việc bổ sung hàm còn thiếu.

Nguồn phương pháp · [Sense and Sensitivity](https://arxiv.org/abs/2505.13353v5), [Code Comprehensor](https://arxiv.org/abs/2610.30016v2), [RepoReasoner](https://arxiv.org/abs/2607.25996), [An Evidence-Centered Structured Survey](https://arxiv.org/abs/2608.21107v1), [A Survey on LLMs for Software Engineering](https://arxiv.org/abs/2312.15223v2).

In [ ]:
import ast
import csv
import getpass
import hashlib
import importlib
import importlib.util
import json
import math
import os
import random
import re
import subprocess
import sys
import time
import types
from collections import Counter, defaultdict
from pathlib import Path

from IPython.display import Markdown, display

if importlib.util.find_spec("tiktoken") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken==0.12.0"])
    importlib.invalidate_caches()
import requests
import tiktoken

SEED = 544013
CASES_PER_FAMILY = 3
DISTRACTOR_MODULES = 64
RETRIEVAL_TOKEN_CAP = 300
TOKENIZER = "o200k_base"       # Ngân sách token tham chiếu, khác tokenizer phía Qwen.
RUN_LLM = True
VTS_CHAT_COMPLETIONS_URL = "https://llm-node3.vts.io.vn/v1/chat/completions"
CHAT_COMPLETIONS_URL = os.environ.get("LLM_CHAT_COMPLETIONS_URL", VTS_CHAT_COMPLETIONS_URL)
MODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen3-VL-8B-Instruct-FP8")
MAX_OUTPUT_TOKENS = 256
OUTPUT_LIMIT_FIELD = "max_tokens"  # Một số máy chủ dùng "max_completion_tokens".
TEMPERATURE = 0
OUTPUT_DIR = Path("ket_qua_5KH")

encoder = tiktoken.get_encoding(TOKENIZER)

def token_count(text):
    return len(encoder.encode_ordinary(text))

assert CASES_PER_FAMILY >= 1 and DISTRACTOR_MODULES >= 3 and RETRIEVAL_TOKEN_CAP > 0

## 1. Tạo dữ liệu và đáp án bằng thực thi

Mỗi ca là một kho mã nhỏ gồm một hàm vào, hai nhánh xử lý, các hàm dùng chung và các đoạn gây nhiễu có tên tương tự. Bộ sinh thay đổi cấu trúc giữa ba họ, đồng thời đổi hằng số và đầu vào theo seed. Vết chạy ghi lại các hàm Python trong kho được gọi với đúng đầu vào của ca. Notebook chỉ thực thi mã do chính bộ sinh bên dưới tạo ra.

In [ ]:
FAMILIES = ("pipeline", "nested", "conditional")

def make_case(family, index, rng):
    a, b, c, d = (rng.randint(11, 79) for _ in range(4))
    x = rng.randint(4, 80) * 2 + index % 2
    if family == "pipeline":
        left = f"    return calibration.offset(value + {a}) * 2"
        right = f"    return calibration.offset(value - {b}) + {c}"
    elif family == "nested":
        left = f"    return calibration.wrap(calibration.offset(value) - {a})"
        right = f"    return calibration.wrap(value + {b}) * 3"
    else:
        left = (f"    if value % 3 == 0:\n"
                f"        return calibration.wrap(value + {a})\n"
                f"    return calibration.offset(value - {b})")
        right = f"    return calibration.wrap(value - {c}) + calibration.offset(value)"
    fold_call = ("getattr(rounding_profile, 'fold')" if family == "nested"
                 else "rounding_profile.fold")
    sources = {
        "entry": (
            "import pricing_routes\n\n"
            "def compute(value):\n"
            "    if value % 2 == 0:\n"
            "        return pricing_routes.primary(value)\n"
            "    return pricing_routes.secondary(value)\n"
        ),
        "pricing_routes": (
            f"import calibration\n\ndef primary(value):\n{left}\n\n"
            f"def secondary(value):\n{right}\n"
        ),
        "calibration": (
            f"import rounding_profile\n\n"
            f"def offset(value):\n    return {fold_call}(value + {d})\n\n"
            f"def wrap(value):\n    return {fold_call}(value * 2 - {b})\n"
        ),
        "rounding_profile": f"def fold(value):\n    return (value * {a} + {c}) % {101 + d}\n",
    }
    # Các đoạn nhiễu dùng thuật ngữ và định danh gần với truy vấn.
    for number in range(DISTRACTOR_MODULES):
        delta = rng.randint(1, 93)
        sources[f"extras_{number:02}"] = (
            f"def primary(value):\n"
            f"    adjusted = (value + {delta}) * {2 + number}\n"
            f"    return adjusted % {103 + number}\n\n"
            f"def secondary(value):\n"
            f"    adjusted = (value - {delta}) * {3 + number}\n"
            f"    return adjusted % {127 + number}\n"
        )
    return {"id": f"{family}-{index:03}", "family": family, "x": x, "sources": sources}

def execute_fixture(case):
    loaded = {}
    real_import = __import__

    def fixture_import(name, globals=None, locals=None, fromlist=(), level=0):
        if not level and name in case["sources"]:
            return load_module(name)
        return real_import(name, globals, locals, fromlist, level)

    def load_module(name):
        if name not in loaded:
            mod = types.ModuleType(name)
            loaded[name] = mod
            mod.__dict__["__builtins__"] = {**vars(importlib.import_module("builtins")), "__import__": fixture_import}
            filename = f"<fixture:{name}>"
            exec(compile(case["sources"][name], filename, "exec"), mod.__dict__)
        return loaded[name]

    entry = load_module("entry")
    active_functions = []

    def capture(frame, event, arg):
        filename = frame.f_code.co_filename
        if event == "call" and filename.startswith("<fixture:"):
            module = filename.removeprefix("<fixture:").removesuffix(">")
            active_functions.append(f"{module}.{frame.f_code.co_name}")

    previous = sys.getprofile()
    sys.setprofile(capture)
    try:
        answer = entry.compute(case["x"])
    finally:
        sys.setprofile(previous)
    return answer, list(dict.fromkeys(active_functions))

rng = random.Random(SEED)
cases = [make_case(family, i, rng) for family in FAMILIES for i in range(CASES_PER_FAMILY)]
for case in cases:
    case["answer"], case["trace"] = execute_fixture(case)
    assert type(case["answer"]) is int and case["trace"][0] == "entry.compute"

print(f"{len(cases)} ca, {len(FAMILIES)} họ chương trình, {len(cases[0]['sources'])} tệp mỗi ca.")
print("Ví dụ vết chạy:", " → ".join(cases[0]["trace"]))

## 2. Phân tích mã và dựng ngữ cảnh

Mỗi đoạn truy xuất tương ứng một hàm, kèm tên tệp và các lệnh import của tệp. Hàm vào và lời gọi với đầu vào cụ thể được cung cấp ở mọi cấu hình. BM25 xếp hạng theo câu hỏi và mã của hàm vào. Biến thể đồ thị ưu tiên các hàm gọi được từ hàm vào hoặc từ kết quả BM25 đứng đầu, theo khoảng cách tối đa ba cạnh. Hai phương pháp dùng cùng bộ đoạn và giới hạn token. Ngữ cảnh chuẩn chỉ dùng cho đối chứng, được dựng từ các hàm đã chạy.

In [ ]:
ENTRY = "entry.compute"
ASK = "Dự đoán giá trị nguyên Python trả về. Chỉ trả về JSON dạng {\"answer\": <số nguyên>}."
SYSTEM = "Bạn đang làm thí nghiệm đọc hiểu mã Python. Suy ra kết quả từ mã nguồn đã cho."

def parse_repository(sources):
    chunks, definitions, imports = {}, {}, {}
    for module, source in sources.items():
        tree = ast.parse(source)
        import_nodes = [node for node in tree.body if isinstance(node, ast.Import)]
        imports[module] = {alias.asname or alias.name: alias.name
                           for node in import_nodes for alias in node.names}
        import_lines = [ast.get_source_segment(source, node) for node in import_nodes]
        for function in (node for node in tree.body if isinstance(node, ast.FunctionDef)):
            key = f"{module}.{function.name}"
            definitions[key] = (module, function)
            body = ast.get_source_segment(source, function)
            chunks[key] = "\n".join([f"# {module}.py :: {function.name}", *import_lines, body])
    graph = {key: set() for key in chunks}
    for key, (module, function) in definitions.items():
        for call in (node.func for node in ast.walk(function) if isinstance(node, ast.Call)):
            destination = None
            if isinstance(call, ast.Name):
                destination = f"{module}.{call.id}"
            elif isinstance(call, ast.Attribute) and isinstance(call.value, ast.Name):
                imported = imports[module].get(call.value.id)
                if imported:
                    destination = f"{imported}.{call.attr}"
            if destination in graph:
                graph[key].add(destination)
    return chunks, graph

def terms(text):
    return re.findall(r"[a-z]+|\d+", text.lower())

def bm25_order(chunks, query, k1=1.5, b=0.75):
    documents = {key: Counter(terms(text)) for key, text in chunks.items() if key != ENTRY}
    df = Counter(term for doc in documents.values() for term in doc)
    lengths = {key: sum(doc.values()) for key, doc in documents.items()}
    avg_length = sum(lengths.values()) / len(lengths)
    query_terms = set(terms(query))
    scores = {}
    for key, document in documents.items():
        norm = k1 * (1 - b + b * lengths[key] / avg_length)
        scores[key] = sum(
            math.log1p((len(documents) - df[term] + 0.5) / (df[term] + 0.5))
            * document[term] * (k1 + 1) / (document[term] + norm)
            for term in sorted(query_terms & document.keys())
        )
    return sorted(documents, key=lambda key: (-scores[key], key))

def expand_order(ranked, graph, hops=3):
    distance = dict.fromkeys([ENTRY, ranked[0]], 0)
    frontier = set(distance)
    for depth in range(1, hops + 1):
        frontier = {neighbor for key in frontier for neighbor in graph[key]} - distance.keys()
        distance.update(dict.fromkeys(frontier, depth))
    return sorted(ranked, key=lambda key: distance.get(key, math.inf))

SEPARATOR = "\n\n"

def context_text(keys, chunks):
    return SEPARATOR.join(chunks[key] for key in keys)

def within_budget(ranked, chunks, budget):
    selected = []
    for key in ranked:
        candidate = [*selected, key]
        if token_count(context_text(candidate, chunks)) <= budget:
            selected = candidate
    return selected

def make_prompt(case, context):
    return (
        f"{ASK}\nLời gọi cần đánh giá: compute({case['x']})\n\n"
        f"Hàm vào (entry.py):\n{case['chunks'][ENTRY]}\n\n"
        f"Các đoạn mã theo tệp:\n\n{context}\n\n"
        "Trả về duy nhất đối tượng JSON."
    )

def build_conditions(case):
    chunks, graph = parse_repository(case["sources"])
    case["chunks"] = chunks
    required = case["trace"][1:]
    query = f"{ASK}\ncompute({case['x']})\n{chunks[ENTRY]}"
    ranked = bm25_order(chunks, query)
    distractors = [key for key in ranked if key not in required]
    starts = {"front": 0, "middle": len(distractors) // 2, "end": len(distractors)}
    selected = {
        f"position_{name}": distractors[:start] + required + distractors[start:]
        for name, start in starts.items()
    }
    selected.update({
        "bm25": within_budget(ranked, chunks, RETRIEVAL_TOKEN_CAP),
        "graph": within_budget(expand_order(ranked, graph), chunks, RETRIEVAL_TOKEN_CAP),
        "oracle": required,
    })
    outputs = {}
    for name, keys in selected.items():
        context = context_text(keys, chunks)
        found = len(set(required) & set(keys))
        prompt = make_prompt(case, context)
        outputs[name] = {
            "case_id": case["id"], "family": case["family"], "condition": name,
            "input": case["x"], "expected": case["answer"], "retrieved": keys,
            "trace": required, "coverage": found / len(required),
            "complete": found == len(required), "context_tokens": token_count(context),
            "prompt_tokens": token_count(prompt), "prompt": prompt,
            "first_evidence_fraction": (
                token_count(context[:context.index(chunks[required[0]])]) / token_count(context)
                if required[0] in keys else None
            ),
        }
    positions = [outputs[f"position_{name}"] for name in starts]
    assert len({row["prompt_tokens"] for row in positions}) == 1, "RQ1: prompt lengths differ."
    assert all(set(row["retrieved"]) == set(ranked) for row in positions)
    assert positions[0]["first_evidence_fraction"] == 0
    assert 0.35 < positions[1]["first_evidence_fraction"] < 0.65
    assert positions[2]["first_evidence_fraction"] > 0.75
    assert all(outputs[name]["context_tokens"] <= RETRIEVAL_TOKEN_CAP for name in ("bm25", "graph"))
    assert outputs["oracle"]["complete"]
    return list(outputs.values())

trials = [row for case in cases for row in build_conditions(case)]
assert len(trials) == len(cases) * 6
assert len({(row["case_id"], row["condition"]) for row in trials}) == len(trials)
trial_order = list(trials)
random.Random(SEED + 1).shuffle(trial_order)

def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

def percent(value):
    return f"{100 * value:.1f}%"

retrieval_names = ("bm25", "graph", "oracle")
retrieval_summary = []
for name in retrieval_names:
    group = [row for row in trials if row["condition"] == name]
    retrieval_summary.append((
        name, len(group), percent(sum(row["complete"] for row in group) / len(group)),
        percent(sum(row["coverage"] for row in group) / len(group)),
        f"{sum(row['context_tokens'] for row in group) / len(group):.1f}",
    ))

print(f"Đã kiểm tra {len(trials)} prompt. Dài nhất: {max(t['prompt_tokens'] for t in trials)} token nội dung.")
print(f"Giới hạn đoạn truy xuất: {RETRIEVAL_TOKEN_CAP} token theo {TOKENIZER}.")
table(["Cấu hình", "Số ca", "Tìm đủ hàm", "Độ bao phủ TB", "Token mã TB"], retrieval_summary)

## 3. Chạy mô hình và lưu từng lần gọi

Notebook dùng khóa từ ô nhập ẩn hoặc biến môi trường VTS_LLM_API_KEY khi chạy endpoint VTS. Nếu đổi máy chủ, điền URL đầy đủ của endpoint Chat Completions và model ở ô cấu hình, rồi nhập khóa của máy chủ đó. Mỗi kết quả được ghi ngay xuống **ket_qua_5KH**. Khi chạy lại với cùng cấu hình và prompt, notebook dùng lại những phản hồi đã lưu. Các lỗi kết nối tạm thời được thử lại tối đa hai lần. Phản hồi sai định dạng vẫn được ghi và tính là trả lời sai. Tên mô hình, thời điểm UTC, số token API trả về, độ trễ và prompt được lưu cùng đáp án. Khóa API chỉ được dùng cho phần xác thực.

In [ ]:
def parse_answer(content):
    text = content.strip()
    fence = re.fullmatch(r"\x60\x60\x60(?:json)?\s*(.*?)\s*\x60\x60\x60", text, re.IGNORECASE | re.DOTALL)
    try:
        result = json.loads(fence.group(1) if fence else text)
    except (ValueError, TypeError):
        return None
    if isinstance(result, dict) and type(result.get("answer")) is int:
        return result["answer"]
    return None

assert parse_answer('{"answer": -9}') == -9
assert parse_answer('{"answer": true}') is None
assert parse_answer('7') is None

def request_answer(prompt, api_key):
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        OUTPUT_LIMIT_FIELD: MAX_OUTPUT_TOKENS,
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
    }
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    started = time.monotonic()
    for attempt in range(3):
        try:
            response = requests.post(
                CHAT_COMPLETIONS_URL, json=payload, headers=headers, timeout=90, allow_redirects=False,
            )
        except (requests.ConnectionError, requests.Timeout):
            if attempt == 2:
                raise
        else:
            if 200 <= response.status_code < 300:
                result = response.json()
                break
            if attempt == 2 or response.status_code not in {408, 429, 500, 502, 503, 504}:
                detail = response.text[:1000]
                if api_key:
                    detail = detail.replace(api_key, "[redacted]")
                raise RuntimeError(f"API HTTP {response.status_code}: {detail}")
        time.sleep(2 ** (attempt + 1))
    choice = result["choices"][0]
    content = choice["message"].get("content") or ""
    return {
        "response": content,
        "served_model": result.get("model"),
        "finish_reason": choice.get("finish_reason"),
        "usage": result.get("usage"),
        "attempts": attempt + 1,
        "latency_seconds": round(time.monotonic() - started, 3),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }

def save_csv(path, rows, fields):
    with path.open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def digest(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()

OUTPUT_DIR.mkdir(exist_ok=True)
dataset_path = OUTPUT_DIR / f"dataset-{digest(cases)[:12]}.json"
dataset_path.write_text(json.dumps(
    [{key: value for key, value in case.items() if key != "chunks"} for case in cases],
    indent=2, ensure_ascii=False,
), encoding="utf-8")

retrieval_path = OUTPUT_DIR / f"retrieval-{digest([cases, RETRIEVAL_TOKEN_CAP, TOKENIZER])[:12]}.csv"
save_csv(retrieval_path, (row for row in trials if row["condition"] in retrieval_names),
         ["case_id", "family", "condition", "coverage", "complete", "context_tokens", "retrieved", "trace"])

results = []
if RUN_LLM:
    if not CHAT_COMPLETIONS_URL.startswith(("https://", "http://")) or not MODEL.strip():
        raise ValueError("Điền CHAT_COMPLETIONS_URL (URL đầy đủ) và MODEL ở ô cấu hình rồi chạy lại.")
    key_variable = "VTS_LLM_API_KEY" if CHAT_COMPLETIONS_URL == VTS_CHAT_COMPLETIONS_URL else "LLM_API_KEY"
    api_key = os.environ.get(key_variable) or getpass.getpass("API key (Enter nếu endpoint không yêu cầu): ")
    if api_key and not CHAT_COMPLETIONS_URL.startswith("https://"):
        from urllib.parse import urlsplit
        if urlsplit(CHAT_COMPLETIONS_URL).hostname not in {"localhost", "127.0.0.1", "::1"}:
            raise ValueError("Dùng HTTPS khi gửi khóa API tới máy chủ từ xa.")
    configuration = {
        "model": MODEL, "url": CHAT_COMPLETIONS_URL, "system": SYSTEM, "temperature": TEMPERATURE,
        "output_limit": MAX_OUTPUT_TOKENS, "output_limit_field": OUTPUT_LIMIT_FIELD,
        "seed": SEED, "cases_per_family": CASES_PER_FAMILY, "tokenizer": TOKENIZER,
        "retrieval_token_cap": RETRIEVAL_TOKEN_CAP, "distractor_modules": DISTRACTOR_MODULES,
    }
    run_id = digest([configuration, trials])[:12]
    (OUTPUT_DIR / f"config-{run_id}.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")
    log_path = OUTPUT_DIR / f"responses-{run_id}.jsonl"
    cache = {}
    if log_path.exists():
        for line in log_path.read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            cache[(record["case_id"], record["condition"])] = record
    with log_path.open("a", encoding="utf-8") as log:
        for index, trial in enumerate(trial_order, 1):
            key = (trial["case_id"], trial["condition"])
            if key not in cache:
                response = request_answer(trial["prompt"], api_key)
                answer = parse_answer(response["response"])
                record = {**trial, **response, "prediction": answer, "correct": answer == trial["expected"]}
                log.write(json.dumps(record, ensure_ascii=False) + "\n")
                log.flush()
                cache[key] = record
            results.append(cache[key])
            if index == 1 or index % 10 == 0 or index == len(trial_order):
                print(f"{index}/{len(trial_order)} ca × cấu hình đã có kết quả.")
else:
    print("RUN_LLM = False. Đã lưu dữ liệu và kết quả truy xuất. Bảng độ chính xác cần chạy mô hình thật.")

## 4. Tổng hợp kết quả theo ba câu hỏi

Báo cáo độ chính xác khớp chính xác giá trị nguyên. Chênh lệch được tính ghép cặp trên cùng ca và biểu diễn bằng điểm phần trăm. Với mẫu tổng hợp nhỏ có cấu trúc lặp theo họ, xem đây là kết quả mô tả. Phân tích RQ3 sử dụng ngữ cảnh chuẩn như một đối chứng chẩn đoán, không dùng vết thực thi để chọn đoạn mã cho BM25 hay đồ thị.

In [ ]:
def summarize(rows, output_path):
    indexed = {(row["case_id"], row["condition"]): row for row in rows}
    case_ids = sorted({row["case_id"] for row in rows})
    condition_names = ("position_front", "position_middle", "position_end", "bm25", "graph", "oracle")
    statistics = []
    for condition in condition_names:
        group = [indexed[case_id, condition] for case_id in case_ids]
        statistics.append({
            "condition": condition,
            "n": len(group),
            "accuracy": sum(row["correct"] for row in group) / len(group),
            "invalid_answers": sum(row["prediction"] is None for row in group),
            "complete_retrieval": sum(row["complete"] for row in group) / len(group),
        })
    table(["Cấu hình", "n", "Đúng", "Sai định dạng", "Tìm đủ hàm"], [
        (row["condition"], row["n"], percent(row["accuracy"]), row["invalid_answers"],
         percent(row["complete_retrieval"]))
        for row in statistics
    ])

    def paired_difference(first, second):
        delta = sum(int(indexed[case_id, first]["correct"]) - int(indexed[case_id, second]["correct"])
                    for case_id in case_ids)
        return f"{100 * delta / len(case_ids):+.1f} điểm %"

    table(["Câu hỏi", "So sánh trên cùng ca", "Chênh lệch đúng"], [
        ("RQ1", "Đầu − Giữa", paired_difference("position_front", "position_middle")),
        ("RQ1", "Cuối − Giữa", paired_difference("position_end", "position_middle")),
        ("RQ2", "Đồ thị − BM25", paired_difference("graph", "bm25")),
    ])

    diagnostics = []
    for condition in ("bm25", "graph"):
        for complete in (False, True):
            wrong = [case_id for case_id in case_ids
                     if indexed[case_id, condition]["complete"] == complete
                     and not indexed[case_id, condition]["correct"]]
            fixed = sum(indexed[case_id, "oracle"]["correct"] for case_id in wrong)
            diagnostics.append({
                "condition": condition, "complete_retrieval": complete, "wrong": len(wrong),
                "fixed_with_oracle": fixed, "still_wrong": len(wrong) - fixed,
            })
    table(["RQ3 · Cấu hình", "Tìm đủ hàm ban đầu", "Sai ban đầu", "Oracle sửa được", "Oracle vẫn sai"], [
        (row["condition"], "Có" if row["complete_retrieval"] else "Thiếu",
         row["wrong"], row["fixed_with_oracle"], row["still_wrong"])
        for row in diagnostics
    ])
    save_csv(output_path / f"summary-{run_id}.csv", statistics, list(statistics[0]))
    save_csv(output_path / f"errors-{run_id}.csv", diagnostics, list(diagnostics[0]))

if results:
    summarize(results, OUTPUT_DIR)

print(f"Thư mục kết quả: {OUTPUT_DIR.resolve()}")